<a href="https://colab.research.google.com/github/kiobov/Analytics/blob/master/decAdhocNotebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Business Problem

Customers are browsing and adding to cart but not buying in month december.

In [ ]:
# mount drive to collab
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive/MyDrive/Kaggle/

In [ ]:
!rm -rf ~/.kaggle
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/Kaggle/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# confirm
!cat ~/.kaggle/kaggle.json

In [ ]:
!pip install kaggle

In [ ]:
!kaggle datasets download -d mkechinov/ecommerce-events-history-in-cosmetics-shop -p /content/drive/MyDrive/

In [ ]:
!ls /content/drive/MyDrive/Kaggle/

In [ ]:
#unzip file
import zipfile
zippath = "/content/drive/MyDrive/ecommerce-events-history-in-cosmetics-shop.zip"
extractpath = "/content/drive/MyDrive/ecommerce-events"

with zipfile.ZipFile(zippath, 'r') as zip_ref:
  zip_ref.extractall(extractpath)


In [ ]:
!pip install duckdb -q

# Understanding data

In [ ]:
import duckdb, pandas as pd

In [ ]:
#open with pandas
#import data
file_path = "/content/drive/MyDrive/ecommerce-events/2019-Dec.csv"
df = pd.read_csv(file_path)

#Overview of the data
print ("Shape: ", df.shape)
print("\nColumns: ", df.columns.tolist())
print("\nSample rows: ")
df.head(3)

In [ ]:
#what unique event types exist
duckdb.query("""
    SELECT event_type, COUNT(*) as total
    FROM df
    GROUP BY event_type
    """).df()

#### Over view possible Conversion Funnel
The concern for the business is people are browsing and adding to cart but not buying. - /

### questions we will look at:
1. Funnel  - of everyone who viewed, how many bought ? Where are people dropping off?
2. Revenue- How much was earned ? Which product/brand drive most revenue?
3. User Behaviour- Are users buying in one session or coming back?
4. Time patterns - When are users most active ? Any peak days?

In [ ]:
#Question 1 - Conversion Funnel,
#Business question: "Out of every 100 people who viewed a product,
#how many ended up buying?"

#Start at counting each stage of the funnel
duckdb.query("""
    SELECT
      event_type,
      COUNT(*) as total_events,
      COUNT(DISTINCT user_id) as unique_users
    FROM df
    WHERE event_type IN ('view', 'cart', 'purchase')
    GROUP BY event_type
    ORDER BY total_events DESC
    """).df()

In [ ]:
# Calculate conversion Rate
duckdb.query("""
    WITH funnel AS (
      SELECT
        event_type,
        COUNT(DISTINCT user_id) as unique_users
      FROM df
      WHERE event_type IN ('view', 'cart', 'purchase')
      GROUP BY event_type
    ),
    totals AS (
      SELECT MAX(CASE WHEN event_type = 'view'  THEN unique_users END) as viewers,
             MAX(CASE WHEN event_type = 'cart'  THEN unique_users END) as carted,
             MAX(CASE WHEN event_type = 'purchase'  THEN unique_users END) as buyers
      FROM funnel
    )
      SELECT
          viewers,
          carted,
          buyers,
          ROUND(carted * 100.0/viewers, 1) AS view_to_cart_pct,
          ROUND(buyers * 100.0/carted, 1) AS cart_to_purchase_pct,
          ROUND(buyers * 100.0/viewers, 1) AS overall_conversion_pct
      FROM totals

    """).df()

### Conversion Overview from the analysis
- 23.3% view - carted meaning roughly 1-4 people who see a product added it to cart. So the product are attratctive enouh to consider

- 30.7%  carted- purchase -  but 7in 10 people whp added to cart never bought (cart abandonment)

- 7.2% overall conversion - not a bad thing however what we need to be vigilant on is (cart abandonment)

### Insights

The problem isnt getting people interested as 23% added to cart. Rtaher the problem is closing the sales. Because for every 10 people who intended to buy 7 leave wthout purchasing

In [ ]:
#Question 2 Revenue
# How much money are we making and what is driving it?
# Total revenue and averange order value

duckdb.query("""
    SELECT
      ROUND(SUM(price), 2) AS total_revenue,
      COUNT(*) AS total_purchases,
      ROUND(AVG(price), 2) AS avg_order_value,
      ROUND(SUM(price)/ COUNT(DISTINCT user_id), 2) AS revenue_per_buyer
    FROM df
    WHERE event_type = 'purchase'
    """).df()

In [ ]:
#top 10 brands by revenue
duckdb.query("""
    SELECT
        brand,
        COUNT(*) AS purchases,
        ROUND(SUM(price), 2) AS total_revenue,
        ROUND(AVG(price), 2) AS avg_price
      FROM df
      WHERE event_type = 'purchase'
      AND brand IS NOT NULL
      GROUP BY brand
      ORDER BY total_revenue DESC
      LIMIT 10
        """).df()

In [ ]:
# top 10 product categogy by revenue
duckdb.query("""
      SELECT
        category_code,
        COUNT(*) AS purchases,
        ROUND(SUM(price), 2) AS total_revenue,
        ROUND(AVG(price), 2) AS avg_price
      FROM df
      WHERE event_type = 'purchase'
            AND category_code IS NOT NULL
      GROUP BY category_code
      ORDER BY total_revenue DESC
      LIMIT 10
        """).df()

### Revenue Overview Insights

With a total revelue of 1, 077, 624 made in december from 21, 317 purchases with an avg of 42/order per buyer meaning we have buyers purchasing more than in avg as single purchase was only $24.85

Brand Analysis

Revenue is driven by high-volume , low -cost nail beauty products. There is one premium brand (strong at 185) but it has few buyers  - either due to being niche or priced out of reach for most users



Category Analysis

The vacuum cleaners generate the most category revenue despite not having the most purchases because the price per unit is high. Gloves sell more units but at  6 dollar each, total revenue <

# Insights
Nail products segment is very strong. with a conversion rate is seem to be owkey on the overall(7.2%) but losses 70% of carters at checkout.

Revenue is healthy at $1M for December , driven by high volume low cost items. The opportunity is in reducing cart abandonment even recovering 10% of abandoned carts would add 100k in revenue

In [ ]:
# Which specific products are being added
#to cart the most but not purchsed
duckdb.query("""
  WITH product_funnel AS (
      SELECT
          product_id,
          brand,
          ROUND(AVG(price), 2) AS price,
          COUNT(CASE WHEN event_type = 'cart' THEN 1 END ) as cart_adds,
          count(CASE WHEN event_type = 'purchase' THEN 1 END )AS purchases
      FROM df
      WHERE brand IS NOT NULL
      GROUP BY product_id , brand
  )
  SELECT
      product_id,
      brand,
      price,
      cart_adds,
      purchases,
      ROUND(purchases * 100.0/ NULLIF(cart_adds, 0), 1) AS purchase_rate_pct
  FROM product_funnel
  WHERE cart_adds >50
  ORDER BY purchase_rate_pct ASC
  LIMIT 15
""").df()

#### Overview

The products costs less than $7 dollars, why are people adding to cart but not buying

#### My hypothesis

  User is comparing options adding multiple cheap once then picking one
  Shipping cost at checkout making a 1 dolar item not worth

  user savin for later but never returning

  Out of stock issues appearing only at checkout

In [ ]:
#quantify revenue leak from product

duckdb.query("""
    WITH product_funnel AS (
        SELECT
            product_id,
            brand,
            ROUND(AVG(price), 2) AS price,
            COUNT(CASE WHEN event_type = 'cart' THEN 1 END) AS cart_adds,
            COUNT(CASE WHEN event_type = 'purchase' THEN 1 END ) AS purchases
        FROM df
        WHERE brand IS NOT NULL
        GROUP BY product_id, brand

    )
    SELECT
        product_id,
        brand,
        price,
        cart_adds,
        purchases,
        cart_adds - purchases AS abandoned_carts,
        ROUND((cart_adds - purchases ) * price, 2 ) AS lost_revenue_usd,
        ROUND(purchases * 100.0 / NULLIF(cart_adds, 0), 1) AS purchase_rate_pct
  FROM product_funnel
  WHERE cart_adds >50
  ORDER BY lost_revenue_usd DESC
  LIMIT 15
""").df()

## Insights
Cart abandonment is costing the business tens of thousands in December alone. The problem is split into 2 segments
  - Expensive items (100- 200 dollars)
where each abandonmenr is costly

  - cheap nail products where volume makes the losses add up. The priority should be fixin checkout for high value items first as ROI per fix is much higher

In [ ]:
#Question 4
#Which days and hours have the most purchases vs most cart abandonment ?

#purchase and abandonment patterns by hour of day
duckdb.query("""
      SELECT
          EXTRACT(HOUR FROM CAST(event_time AS TIMESTAMP))  AS hour_of_day,
          COUNT(CASE WHEN event_type = 'cart' THEN 1 END ) AS cart_adds,
          COUNT(CASE WHEN event_type = 'purchase' THEN 1 END ) AS purchases,
          ROUND(
            COUNT(CASE WHEN event_type = 'purchase' THEN 1 END) * 100.0/
            NULLIF(COUNT(CASE WHEN event_type = 'cart' THEN 1 END ), 0), 1
          ) AS conversion_pct

      FROM df
      GROUP BY hour_of_day
      ORDER BY hour_of_day
""").df()

In [ ]:
#which day of the week performs best
duckdb.query("""
      SELECT
          DAYNAME(CAST(event_time AS TIMESTAMP)) AS day_of_week,
          COUNT(CASE WHEN event_type = 'view'  THEN 1 END ) AS views,
          COUNT(CASE WHEN event_type = 'vart' THEN 1 END ) AS cart_adds,
          COUNT(CASE WHEN event_type = 'purchase' THEN 1 END ) AS purchases,
          ROUND(
              COUNT(CASE WHEN  event_type = 'purchase' THEN 1 END ) * 100.0/
              NULLIF(COUNT(CASE WHEN event_type = 'cart' THEN 1 END ), 0), 1
          ) AS conversion_pct
      FROM df
      GROUP BY day_of_week
      ORDER BY purchases DESC

      """).df()

# Business Insights

The highest value customers shop on Thursday and friday morning between 9am - 12pm.

This iw when Promotions , emails campaigns and retargeting ads will get the best returns.

Evening cart abandones 7pm - 9pm , especially Sunday) are the biggest recovery opportunity - a timed reminder email sent the following morning at 10am could recover a significat portion of that lost revenue

In [ ]:
#Question 5
#Are we relying on a small group of heavy buyers,
#or dowe have a heathy spread of customers ?

#How many purchases does each buyer make?
duckdb.query("""
      WITH user_purchases AS (
          SELECT
              user_id,
              COUNT(*) AS total_purchases,
              ROUND(SUM(price), 2 ) AS total_spent
          FROM df
          WHERE event_type = 'purchase'
          GROUP BY user_id
      )
      SELECT
          total_purchases,
          COUNT(user_id) AS num_users,
          ROUND(AVG(total_spent), 2) AS avg_spent
      FROM user_purchases
      GROUP BY total_purchases
      ORDER BY total_purchases
      LIMIT 15
""").df()
# Top 10 highest spending users


In [ ]:
#top 10 highest spending users
duckdb.query("""
        SELECT
            user_id,
            COUNT(*) AS total_purchases,
            ROUND(SUM(price), 2) AS total_spent,
            ROUND(AVG(price), 2) AS avg_order_value
        FROM df
        WHERE event_type = 'purchase'
        GROUP BY user_id
        ORDER BY total_spent DESC
        LIMIT 10

            """).df()

### Top Spender Analysis
The top buyers split ontp 2 types :
- bulk buyers of cheap nail products (likely trader/ reseller)

-  occassional buyers of mid -range items. Neither profile resemble a typical retail consumer

# Report December 2019

Store profile - with over 3.5 user events, serving a mix of retail consumers and likely trade/reseller buyers.

Funnnel Health - The store converts at 7.2% overall above previous months but losses 70% of users ath the cart to purchase stage. For every 10 people who intend to buy , 7 leave without completing the purchase.

Revenue December revenue was 1M across 21, 317 purchases. high volume , low cost nail products drive most transactions , but high ticket items like (strong ) (185-194) and marathon (137) create the largest individual revenue losses when abandoned

Biggest Opportunity - Cary abandonment across just 15 products represents over $200k in recover revenue . High-price fix - each recovered sale is worth 100-194

When to act Peak buying happens Thursday - friday 9am-12pm. Evening browsers (7-9pm especially sunday) add to cart but dont buy a morning follow up reminder to these users represent the single highest ROI marketing action available

Customer Loyalty Risk 50% of buyers purchased only once. The business is heavily acquision -dependent.A loyalty or repeat purchase incentive program targeting the 12, 910 one time buyers could significantly increase revenue without any new customer acquisition cost

In [ ]:
#